# CI-Triage-Env — GRPO Training

**Hardware target**: A10G Large (46 GB VRAM, 12 vCPU) — HuggingFace Space

Pipeline:
1. Install dependencies
2. Authenticate (HF + W&B)
3. Pull scenario corpus from HF Hub
4. Pull SFT dataset from HF Hub
5. SFT warmstart (~45 min)
6. GRPO fine-tuning (~90 min for 100 steps)
7. Push final model to HF Hub

**Set these as Space secrets** (Settings → Variables and secrets):
- `HF_TOKEN` — HuggingFace write token
- `HF_USERNAME` — your HF username
- `WANDB_API_KEY` — Weights & Biases key (get free at wandb.ai)

**Time budget**: SFT≈45 min + GRPO≈90 min = ~2.5 hours on A10G Large.
Monitor training live at https://wandb.ai (project: `ci-triage-env`).

In [ ]:
# Cell 1 — Install deps (run once; ~10 min including unsloth compile)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return result.stdout

# PyTorch is pre-installed in the Space Docker image; install the rest
run('pip install -q "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q trl>=0.11 transformers>=4.45 accelerate>=0.30 peft')
run('pip install -q wandb datasets huggingface_hub')
run('pip install -q -e /workspace')  # install ci_triage_env package
print('All dependencies installed.')

In [ ]:
# Cell 2 — Authenticate
import os
from huggingface_hub import login
import wandb

HF_TOKEN    = os.environ['HF_TOKEN']
HF_USERNAME = os.environ['HF_USERNAME']
WANDB_KEY   = os.environ.get('WANDB_API_KEY', '')

login(token=HF_TOKEN)
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    os.environ['WANDB_PROJECT'] = 'ci-triage-env'
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled — set WANDB_API_KEY secret to enable')

# Repo names (edit if you used different names)
SCENARIOS_REPO   = f'{HF_USERNAME}/ci-triage-scenarios'
SFT_DATASET_REPO = f'{HF_USERNAME}/ci-triage-sft'
MODEL_REPO       = f'{HF_USERNAME}/ci-triage-agent'
print(f'Authenticated as {HF_USERNAME}')

In [ ]:
# Cell 3 — Download scenario corpus from HF Hub
from pathlib import Path
from huggingface_hub import snapshot_download

SCEN_DIR = Path('/data/scenarios')
SCEN_DIR.mkdir(parents=True, exist_ok=True)

existing = list(SCEN_DIR.rglob('*.json'))
if existing:
    print(f'Scenarios already present: {len(existing)} files — skipping download')
else:
    snapshot_download(
        repo_id=SCENARIOS_REPO,
        repo_type='dataset',
        local_dir=str(SCEN_DIR),
        token=HF_TOKEN,
    )
    n = len(list(SCEN_DIR.rglob('*.json')))
    print(f'Downloaded {n} scenario files')

train_dir = SCEN_DIR / 'train'
print(f'Train scenarios: {len(list(train_dir.rglob("*.json")))}')

In [ ]:
# Cell 4 — Download SFT dataset from HF Hub
from datasets import load_dataset, load_from_disk

SFT_DS_DIR = Path('/data/sft_dataset')

if SFT_DS_DIR.exists():
    ds = load_from_disk(str(SFT_DS_DIR))
    print(f'SFT dataset already present: {len(ds)} examples')
else:
    ds = load_dataset(SFT_DATASET_REPO, split='train', token=HF_TOKEN)
    SFT_DS_DIR.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(SFT_DS_DIR))
    print(f'Downloaded {len(ds)} SFT examples')

In [ ]:
# Cell 5 — SFT warmstart
# Estimated time: ~45 min on A10G Large
# Optimised: batch_size=4, grad_accum=4 → effective batch 16, bf16 via unsloth
from ci_triage_env.training.sft import run_sft

SFT_CKPT = Path('/data/checkpoints/sft')

if SFT_CKPT.exists():
    print(f'SFT checkpoint found at {SFT_CKPT} — skipping (delete to retrain)')
else:
    run_sft(
        dataset_path=str(SFT_DS_DIR),
        output_dir=str(SFT_CKPT),
        num_epochs=2,
        per_device_batch_size=4,       # A10G Large has 46 GB — fits 4 sequences
        gradient_accumulation_steps=4, # effective batch = 16
    )
    print(f'SFT done → {SFT_CKPT}')

    # Push immediately so checkpoint is safe even if GRPO fails
    from huggingface_hub import upload_folder
    upload_folder(
        folder_path=str(SFT_CKPT),
        repo_id=MODEL_REPO + '-sft',
        repo_type='model',
        token=HF_TOKEN,
        commit_message='SFT warmstart checkpoint (Qwen3.5-4B + LoRA)',
    )
    print(f'SFT checkpoint pushed to {MODEL_REPO}-sft')

In [ ]:
# Cell 6 — GRPO fine-tuning
# Estimated time: ~90 min for 100 steps on A10G Large
#
# Why 100 steps? Each step = 4 multi-turn rollouts (max 4 tool calls each).
# Sequential rollout with model.generate() is the bottleneck: ~50 sec/step.
# Increase GRPO_STEPS if you have more time budget.
#
# MockEnvClient is used in-process — no server needed, full speed.

from ci_triage_env.training.mock_env_client import MockEnvClient
from ci_triage_env.training.grpo import run_grpo

GRPO_CKPT  = Path('/data/checkpoints/grpo')
GRPO_STEPS = 100  # increase to 200 if you have ~3 hours total

env_client = MockEnvClient(scenarios_dir=str(SCEN_DIR / 'train'))
print(f'MockEnvClient loaded {len(env_client.scenario_ids)} train scenarios')
print(f'Starting GRPO — {GRPO_STEPS} steps, ~{GRPO_STEPS * 50 // 60} min estimated')
print('Monitor: https://wandb.ai (project: ci-triage-env)')

run_grpo(
    sft_checkpoint_dir=str(SFT_CKPT),
    output_dir=str(GRPO_CKPT),
    total_steps=GRPO_STEPS,
    env_client=env_client,
    scenarios_train_path=str(SCEN_DIR / 'train'),
    hyperparams={
        # ── training update (fast) ──────────────────────
        'per_device_train_batch_size': 1,
        'gradient_accumulation_steps': 4,   # effective batch = 4
        'learning_rate': 5e-6,
        'kl_coef': 0.04,
        # ── rollout generation (bottleneck) ────────────
        'num_generations': 4,               # 4 rollouts per training sample
        'max_prompt_length': 2048,
        'max_completion_length': 256,        # short = fast; CI responses are concise
        'temperature': 0.8,
        'top_p': 0.95,
        # ── logging ────────────────────────────────────
        'logging_steps': 5,
        'save_steps': 50,
        'report_to': 'wandb' if WANDB_KEY else 'none',
    },
)
print(f'GRPO done → {GRPO_CKPT}')

In [ ]:
# Cell 7 — Push final model to HF Hub
from huggingface_hub import upload_folder

upload_folder(
    folder_path=str(GRPO_CKPT),
    repo_id=MODEL_REPO,
    repo_type='model',
    token=HF_TOKEN,
    commit_message=f'GRPO-trained adapter — {GRPO_STEPS} steps on A10G Large',
)
print(f'Final model: https://huggingface.co/{MODEL_REPO}')